In [ ]:
# @title
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import time

# ==========================================
# HELPER FUNCTION FOR SAFE USER INPUT
# ==========================================
def get_user_input(prompt, default_val, cast_type=int):
    user_str = input(f"{prompt} [Default: {default_val}]: ").strip()
    if not user_str:
        return default_val
    try:
        return cast_type(user_str)
    except ValueError:
        print(f"Invalid input. Using default value: {default_val}")
        return default_val

# ==========================================
# 1. COLLECT USER INPUTS
# ==========================================
def configure_from_user():
    print("=== SOLAR PANEL PLACEMENT OPTIMIZATION (PCA) ===")
    print("Enter parameters below (or press Enter to use default values):\n")

    rows = get_user_input("1. Rooftop Grid Rows (e.g., 8 to 20)", 10, int)
    cols = get_user_input("2. Rooftop Grid Columns (e.g., 8 to 20)", 10, int)
    max_iter = get_user_input("3. Maximum PCA Iterations", 50, int)
    panel_cost = get_user_input("4. Panel Cost/Maintenance Penalty (0.1 to 0.5)", 0.25, float)
    shadow_loss = get_user_input("5. Adjacent Shadow Interference Penalty (0.2 to 0.9)", 0.60, float)

    print("\n--- Obstacle / Permanent Shadow Zone (e.g., Water Tank / Chimney) ---")
    add_obstacle = input("Do you want to define a shaded obstacle area? (y/n) [Default: y]: ").strip().lower()

    obstacle_coords = None
    if add_obstacle != 'n':
        obs_r_start = get_user_input(f"   Obstacle Start Row (0 to {rows-1})", min(2, rows-1), int)
        obs_r_end = get_user_input(f"   Obstacle End Row ({obs_r_start} to {rows-1})", min(4, rows-1), int)
        obs_c_start = get_user_input(f"   Obstacle Start Col (0 to {cols-1})", min(2, cols-1), int)
        obs_c_end = get_user_input(f"   Obstacle End Col ({obs_c_start} to {cols-1})", min(4, cols-1), int)
        obstacle_coords = (obs_r_start, obs_r_end, obs_c_start, obs_c_end)

    return rows, cols, max_iter, panel_cost, shadow_loss, obstacle_coords

# ==========================================
# 2. CELLULAR FITNESS EVALUATION FUNCTION
# ==========================================
def evaluate_cell_fitness(args):
    r, c, current_grid, sun_map, rows, cols, panel_cost, shadow_penalty = args

    if current_grid[r, c] == 0:
        return (r, c, 0.0)

    base_energy = sun_map[r, c]
    shadow_loss = 0.0

    # Direct shadow cast from southern neighboring panel (row + 1)
    if r + 1 < rows and current_grid[r + 1, c] == 1:
        shadow_loss += shadow_penalty

    # Minor crowding/heat interference from left and right neighbors
    if c - 1 >= 0 and current_grid[r, c - 1] == 1:
        shadow_loss += 0.10
    if c + 1 < cols and current_grid[r, c + 1] == 1:
        shadow_loss += 0.10

    net_fitness = base_energy - shadow_loss - panel_cost
    return (r, c, round(net_fitness, 4))


def evaluate_grid_parallel(current_grid, sun_map, rows, cols, panel_cost, shadow_penalty):
    tasks = [
        (r, c, current_grid, sun_map, rows, cols, panel_cost, shadow_penalty)
        for r in range(rows)
        for c in range(cols)
    ]
    fitness_grid = np.zeros((rows, cols))

    with ThreadPoolExecutor() as executor:
        results = executor.map(evaluate_cell_fitness, tasks)
        for r, c, fit in results:
            fitness_grid[r, c] = fit

    return fitness_grid

# ==========================================
# 3. SYNCHRONOUS NEIGHBORHOOD UPDATE RULE
# ==========================================
def update_grid_synchronous(current_grid, fitness_grid, sun_map, rows, cols, panel_cost, mutation_rate=0.05):
    next_grid = current_grid.copy()

    for r in range(rows):
        for c in range(cols):
            r_min, r_max = max(0, r - 1), min(rows, r + 2)
            c_min, c_max = max(0, c - 1), min(cols, c + 2)

            local_fitness = fitness_grid[r_min:r_max, c_min:c_max]
            current_state = current_grid[r, c]
            current_fit = fitness_grid[r, c]

            # Remove unprofitable/shaded panels
            if current_state == 1 and current_fit < 0:
                next_grid[r, c] = 0

            # Install panel in sunny unshaded spots influenced by thriving neighbors
            elif current_state == 0:
                best_neighbor_fit = np.max(local_fitness)
                if sun_map[r, c] > (panel_cost + 0.25) and best_neighbor_fit > 0.15:
                    south_shaded = (r + 1 < rows and current_grid[r + 1, c] == 1)
                    if not south_shaded:
                        next_grid[r, c] = 1

            # Small random exploration
            if np.random.rand() < mutation_rate:
                next_grid[r, c] = 1 - next_grid[r, c]

    return next_grid

# ==========================================
# 4. MAIN EXECUTION LOOP
# ==========================================
def run_pca():
    rows, cols, max_iter, panel_cost, shadow_penalty, obstacle_coords = configure_from_user()

    np.random.seed(42)

    # Generate sunlight exposure map (0.5 to 1.0 across roof)
    sunlight_map = np.random.uniform(0.5, 1.0, size=(rows, cols))

    # Apply user-defined obstacle shadow zone
    if obstacle_coords:
        r1, r2, c1, c2 = obstacle_coords
        sunlight_map[r1:r2+1, c1:c2+1] = 0.05  # Heavy shade / unbuildable zone

    # Initial random solar panel placement
    grid = np.random.choice([0, 1], size=(rows, cols), p=[0.6, 0.4])

    print("\n" + "="*50)
    print("RUNNING PARALLEL CELLULAR OPTIMIZATION...")
    print("="*50)

    initial_fitness = evaluate_grid_parallel(grid, sunlight_map, rows, cols, panel_cost, shadow_penalty)
    print(f"Initial Total Panels : {np.sum(grid)}")
    print(f"Initial Net Energy   : {np.sum(initial_fitness):.2f} units\n")

    best_grid = grid.copy()
    best_total_energy = np.sum(initial_fitness)

    start_time = time.time()

    for iteration in range(1, max_iter + 1):
        fitness_grid = evaluate_grid_parallel(grid, sunlight_map, rows, cols, panel_cost, shadow_penalty)
        total_energy = np.sum(fitness_grid)

        if total_energy > best_total_energy:
            best_total_energy = total_energy
            best_grid = grid.copy()

        new_grid = update_grid_synchronous(grid, fitness_grid, sunlight_map, rows, cols, panel_cost)

        if np.array_equal(grid, new_grid):
            print(f"Converged early at Iteration {iteration}!")
            break

        grid = new_grid

    elapsed = time.time() - start_time

    print(f"Optimization Completed in {elapsed:.3f} seconds.")
    print(f"Optimized Total Panels : {np.sum(best_grid)}")
    print(f"Optimized Net Energy   : {best_total_energy:.2f} units\n")

    print("Optimized Rooftop Layout (1 = Panel, X = Shaded Obstacle, . = Empty):")
    for r in range(rows):
        row_cells = []
        for c in range(cols):
            if sunlight_map[r, c] <= 0.1:
                row_cells.append("X")
            elif best_grid[r, c] == 1:
                row_cells.append("1")
            else:
                row_cells.append(".")
        print(f"Row {r:02d}: " + " ".join(row_cells))

if __name__ == "__main__":
    run_pca()


In [6]:
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import time

# ==========================================
# 1. CONFIGURATION & PROBLEM PARAMETERS
# ==========================================
GRID_ROWS = 5
GRID_COLS = 5
MAX_ITERATIONS = 20
PANEL_COST_PENALTY = 0.30   # Penalty per panel (installation/maintenance cost)
SHADOW_PENALTY = 0.80       # Energy loss when shaded by a south/adjacent panel
MUTATION_RATE = 0.15        # Probability of exploring a new placement

# Set random seed for reproducible results
np.random.seed(42)

# ==========================================
# 2. INITIALIZE ROOFTOP ENVIRONMENT
# ==========================================
# Sunlight exposure map (0.2 = heavy shade near chimney/walls, 1.0 = full sun)
sunlight_map = np.random.uniform(0.4, 1.0, size=(GRID_ROWS, GRID_COLS))

# Simulate a shaded obstacle area (e.g., HVAC unit or parapet wall shadow)
sunlight_map[2:4, 2:4] = 0.1

# Initial random solar panel placement (0 = Empty, 1 = Solar Panel)
grid = np.random.choice([0, 1], size=(GRID_ROWS, GRID_COLS), p=[0.6, 0.4])


# ==========================================
# 3. CELLULAR FITNESS EVALUATION FUNCTION
# ==========================================
def evaluate_cell_fitness(args):
    """
    Evaluates the net energy fitness of a single rooftop cell (r, c)
    considering its sunlight exposure and shadow cast by immediate neighbors.
    """
    r, c, current_grid, sun_map = args

    # If there is no panel at this cell, net energy generated is 0
    if current_grid[r, c] == 0:
        return (r, c, 0.0)

    base_energy = sun_map[r, c]
    shadow_loss = 0.0

    # Check immediate neighbors (3x3 Moore Neighborhood) for shadow interference
    # Suppose sun comes primarily from the South (row + 1) and East/West
    if r + 1 < GRID_ROWS and current_grid[r + 1, c] == 1:
        shadow_loss += SHADOW_PENALTY  # Direct shadow from southern panel

    # Minor crowding/heat interference from adjacent left/right panels
    if c - 1 >= 0 and current_grid[r, c - 1] == 1:
        shadow_loss += 0.10
    if c + 1 < GRID_COLS and current_grid[r, c + 1] == 1:
        shadow_loss += 0.10

    net_fitness = base_energy - shadow_loss - PANEL_COST_PENALTY
    return (r, c, round(net_fitness, 4))


def evaluate_grid_parallel(current_grid, sun_map):
    """Evaluates all cells in the grid concurrently."""
    tasks = [
        (r, c, current_grid, sun_map)
        for r in range(GRID_ROWS)
        for c in range(GRID_COLS)
    ]
    fitness_grid = np.zeros((GRID_ROWS, GRID_COLS))

    with ThreadPoolExecutor() as executor:
        results = executor.map(evaluate_cell_fitness, tasks)
        for r, c, fit in results:
            fitness_grid[r, c] = fit

    return fitness_grid


# ==========================================
# 4. NEIGHBORHOOD INTERACTION & UPDATE RULE
# ==========================================
def update_grid_synchronous(current_grid, fitness_grid, sun_map):
    """
    Each cell inspects its 3x3 neighborhood and updates its state simultaneously.
    """
    next_grid = current_grid.copy()

    for r in range(GRID_ROWS):
        for c in range(GRID_COLS):
            # Extract 3x3 neighborhood bounds
            r_min, r_max = max(0, r - 1), min(GRID_ROWS, r + 2)
            c_min, c_max = max(0, c - 1), min(GRID_COLS, c + 2)

            local_fitness = fitness_grid[r_min:r_max, c_min:c_max]
            local_sun = sun_map[r_min:r_max, c_min:c_max]

            current_state = current_grid[r, c]
            current_fit = fitness_grid[r, c]

            # Rule 1: If panel has negative net fitness (shaded/unprofitable), remove it
            if current_state == 1 and current_fit < 0:
                next_grid[r, c] = 0

            # Rule 2: If cell is empty, check if it has high sunlight & profitable neighbors
            elif current_state == 0:
                best_neighbor_fit = np.max(local_fitness)
                # Place a panel if sunlight here is good and a neighbor is thriving
                if sun_map[r, c] > (PANEL_COST_PENALTY + 0.25) and best_neighbor_fit > 0.2:
                    # Ensure placing a panel won't immediately get shaded by South neighbor
                    south_shaded = (r + 1 < GRID_ROWS and current_grid[r + 1, c] == 1)
                    if not south_shaded:
                        next_grid[r, c] = 1

            # Rule 3: Small random mutation to escape local optima
            if np.random.rand() < MUTATION_RATE:
                next_grid[r, c] = 1 - next_grid[r, c]

    return next_grid


# ==========================================
# 5. MAIN PARALLEL CELLULAR ALGORITHM LOOP
# ==========================================
def run_pca():
    global grid
    print("=== SOLAR PANEL PLACEMENT OPTIMIZATION (PCA) ===")

    initial_fitness = evaluate_grid_parallel(grid, sunlight_map)
    print(f"Initial Total Panels : {np.sum(grid)}")
    print(f"Initial Net Energy   : {np.sum(initial_fitness):.2f} units\n")

    best_grid = grid.copy()
    best_total_energy = np.sum(initial_fitness)

    start_time = time.time()

    for iteration in range(1, MAX_ITERATIONS + 1):
        # Step A: Evaluate fitness of all cells in parallel
        fitness_grid = evaluate_grid_parallel(grid, sunlight_map)
        total_energy = np.sum(fitness_grid)

        # Track global best configuration
        if total_energy > best_total_energy:
            best_total_energy = total_energy
            best_grid = grid.copy()

        # Step B: Synchronous cellular state update based on neighbors
        new_grid = update_grid_synchronous(grid, fitness_grid, sunlight_map)

        # Check early convergence
        if np.array_equal(grid, new_grid):
            print(f"Converged early at Iteration {iteration}!")
            break

        grid = new_grid

    elapsed = time.time() - start_time

    print(f"Optimization Finished in {elapsed:.3f} seconds.")
    print(f"Optimized Total Panels : {np.sum(best_grid)}")
    print(f"Optimized Net Energy   : {best_total_energy:.2f} units\n")

    print("Optimized Rooftop Layout (1 = Solar Panel, . = Empty Space):")
    for r in range(GRID_ROWS):
        row_str = " ".join(["1" if best_grid[r, c] == 1 else "." for c in range(GRID_COLS)])
        print(f"Row {r:02d}: {row_str}")

if __name__ == "__main__":
    run_pca()


=== SOLAR PANEL PLACEMENT OPTIMIZATION (PCA) ===
Initial Total Panels : 8
Initial Net Energy   : -1.72 units

Optimization Finished in 0.036 seconds.
Optimized Total Panels : 9
Optimized Net Energy   : 2.98 units

Optimized Rooftop Layout (1 = Solar Panel, . = Empty Space):
Row 00: 1 1 . . .
Row 01: . . 1 . .
Row 02: 1 . . . 1
Row 03: . 1 . . .
Row 04: 1 . . 1 1
